In [ ]:
from collections import defaultdict, deque

class Assignment:
    def __init__(self, a_id, in1, in2, out, food):
        self.id = f"A{a_id}"
        self.inputs = [int(in1), int(in2)]
        self.out = int(out)
        self.food = food

def compute_depths(assignments, adj_list):
    memo = {}
    def dfs(node):
        if node in memo:
            return memo[node]

        max_depth = 0

        # If the node has neighbors, calculate the max depth among them
        if adj_list[node]:
            for neighbor in adj_list[node]:
                max_depth = max(max_depth, 1 + dfs(neighbor))

        memo[node] = max_depth
        return max_depth

    # Run DFS for every assignment to populate the dictionary
    for a in assignments:
        dfs(a.id)
    return memo


def solve_greedy(assignments, food_costs, initial_inputs, g, strategy="food_cost"):
    """ Strategies: 'food_cost' or 'dependency_depth' """

    out_to_assign = {a.out: a.id for a in assignments}
    assign_map = {a.id: a for a in assignments}

    in_degree = {a.id: 0 for a in assignments}
    adj_list = defaultdict(list)

    for a in assignments:
        for req in a.inputs:
            if req not in initial_inputs:
                if req in out_to_assign:
                    parent = out_to_assign[req]
                    adj_list[parent].append(a.id)
                    in_degree[a.id] += 1

    depths = compute_depths(assignments, adj_list) if strategy == "dependency_depth" else {}

    available = [a.id for a in assignments if in_degree[a.id] == 0]

    schedule = []
    total_cost = 0

    day = 1
    while available:
        if strategy == "food_cost":
            available.sort(key=lambda x: (food_costs[assign_map[x].food], x))

        elif strategy == "dependency_depth":
            available.sort(key=lambda x: (-depths[x], x))

        todays_tasks = available[:g]
        available = available[g:] 
        

        todays_menu = defaultdict(int)
        day_cost = 0

        for task in todays_tasks:
            food_item = assign_map[task].food
            todays_menu[food_item] += 1
            day_cost += food_costs[food_item]

            for neighbor in adj_list[task]:
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    available.append(neighbor)

        schedule.append({
            "day": day,
            "tasks": todays_tasks,
            "menu": dict(todays_menu),
            "cost": day_cost
        })
        total_cost += day_cost
        day += 1

    return schedule, total_cost



if __name__ == "__main__":
    food_costs = {'TC': 1, 'DF': 1, 'PM': 1, 'GJ': 1}
    g = 2
    initial_inputs = {1, 2, 3, 4, 5, 6}

    raw_assignments = [
        (1, 1, 3, 7, 'TC'), (2, 4, 2, 8, 'TC'), (3, 1, 3, 9, 'TC'),
        (4, 2, 3, 10, 'PM'), (5, 7, 8, 11, 'TC'), (6, 4, 6, 12, 'TC'),
        (7, 6, 9, 13, 'PM'), (8, 10, 5, 14, 'GJ'), (9, 1, 11, 15, 'DF'),
        (10, 3, 12, 16, 'TC'), (11, 15, 16, 17, 'DF')
    ]

    assignments = [Assignment(*data) for data in raw_assignments]

    print("=== Strategy 1: Greedy by Food Cost ===")
    sched_food, cost_food = solve_greedy(assignments, food_costs, initial_inputs, g, "food_cost")
    for day in sched_food:
        print(f"Day-{day['day']}: {', '.join(day['tasks'])}")
        menu_str = ", ".join([f"{v}-{k}" for k, v in day['menu'].items()])
        print(f"Menu: {menu_str} | Cost: {day['cost']}")
    print(f"Total Days: {len(sched_food)} | Total Cost: {cost_food}\n")


    print("=== Strategy 2: Greedy by Dependency Depth ===")
    sched_depth, cost_depth = solve_greedy(assignments, food_costs, initial_inputs, g, "dependency_depth")
    for day in sched_depth:
        print(f"Day-{day['day']}: {', '.join(day['tasks'])}")
        menu_str = ", ".join([f"{v}-{k}" for k, v in day['menu'].items()])
        print(f"Menu: {menu_str} | Cost: {day['cost']}")
    print(f"Total Days: {len(sched_depth)} | Total Cost: {cost_depth}")

=== Strategy 1: Greedy by Food Cost ===
Day-1: A1, A2
Menu: 2-TC | Cost: 2
Day-2: A3, A4
Menu: 1-TC, 1-PM | Cost: 2
Day-3: A5, A6
Menu: 2-TC | Cost: 2
Day-4: A10, A7
Menu: 1-TC, 1-PM | Cost: 2
Day-5: A8, A9
Menu: 1-GJ, 1-DF | Cost: 2
Day-6: A11
Menu: 1-DF | Cost: 1
Total Days: 6 | Total Cost: 11

=== Strategy 2: Greedy by Dependency Depth ===
Day-1: A1, A2
Menu: 2-TC | Cost: 2
Day-2: A5, A6
Menu: 2-TC | Cost: 2
Day-3: A10, A3
Menu: 2-TC | Cost: 2
Day-4: A4, A9
Menu: 1-PM, 1-DF | Cost: 2
Day-5: A11, A7
Menu: 1-DF, 1-PM | Cost: 2
Day-6: A8
Menu: 1-GJ | Cost: 1
Total Days: 6 | Total Cost: 11
